# Name That LLM — first submission

A reproducible CPU baseline: **word + character TF-IDF → class-balanced linear SVM**.
No downloads, API calls, external datasets, or GPU are needed. Run cells in order.

## Competition requirements checked on 30 October 2026

The [official overview](https://www.kaggle.com/competitions/name-that-llm/overview/citation)
specifies balanced accuracy (mean recall across 18 models), an offline notebook,
a maximum nine-hour CPU/GPU runtime, and output named `submission.parquet`.
Output has exactly `id` and `model`, one valid model name per test ID.
Interactive test data has one example; scoring reruns this code on the hidden test.
Never hardcode the example's size or predictions.

**Access limitation:** the [full Rules page](https://www.kaggle.com/competitions/name-that-llm/rules)
was blocked by reCAPTCHA during preparation. The [Data page](https://www.kaggle.com/competitions/name-that-llm/data)
did not expose the file contents. This is not a claim that the complete legal rules were verified.
Read and accept them in your Kaggle account before submitting. The notebook uses only
competition training data and checks its expected schema at runtime.
No real-data score is claimed; preparation checks used synthetic fixtures.

## 1. Settings and dependencies

Kaggle's Python environment normally includes these libraries. Keep Internet **off**
and choose **CPU / no accelerator**. There are no installation cells.
Locally, install `numpy pandas scipy scikit-learn pyarrow matplotlib ipython` first.
Set `DATA_DIR` to the directory containing your authorized competition files.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
import gc
import importlib.util
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

SEED = 42
N_SPLITS = 5
RUN_CV = True                    # Keep True for a useful first validation result.
C = 1.0                         # Fixed starting value; avoid tuning on leaderboard scores.
BATCH_SIZE = 512                 # Bounds inference memory on the hidden test.
DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
TEXT_COLUMN = None              # Auto-detect text / text_snippet; set explicitly if needed.

VALID_LABELS = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5',
    'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3',
]
assert len(VALID_LABELS) == len(set(VALID_LABELS)) == 18
np.random.seed(SEED)
print(f'pandas={pd.__version__}, scikit-learn={sklearn.__version__}')

## 2. Read and validate the data

Expected files: `train.csv` (`id`, `prompt_id`, `text`, `model`) and
`test.csv` (`id`, `text`); `sample_submission.parquet` is an optional format reference.
These input column names are checked rather than silently guessed. `text_snippet`
is also supported. CSV parsing handles quoted commas and newlines in passages.
IDs stay strings, including leading zeros. We retain punctuation, Unicode and case:
these may carry authorship information. Neither IDs nor prompt IDs are model inputs.

In [ ]:
# Some Kaggle notebooks mount competition data at the older, shorter path.
if not (DATA_DIR / 'train.csv').is_file():
    alternate = Path('/kaggle/input/name-that-llm')
    if DATA_DIR == Path('/kaggle/input/competitions/name-that-llm') and (alternate / 'train.csv').is_file():
        DATA_DIR = alternate

for filename in ['train.csv', 'test.csv']:
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f'{DATA_DIR / filename}: attach the competition data or change DATA_DIR.')

# keep_default_na=False preserves literal strings such as "NA" and "null".
train = pd.read_csv(DATA_DIR / 'train.csv', dtype=str, keep_default_na=False)
test = pd.read_csv(DATA_DIR / 'test.csv', dtype=str, keep_default_na=False)
print('Train:', train.shape, train.columns.tolist())
print('Test: ', test.shape, test.columns.tolist())

if TEXT_COLUMN is None:
    candidates = [c for c in ['text', 'text_snippet'] if c in train and c in test]
    if len(candidates) != 1:
        raise ValueError(f'Set TEXT_COLUMN explicitly. Shared text candidates: {candidates}')
    TEXT_COLUMN = candidates[0]

for name, frame, required in [
    ('train', train, {'id', 'prompt_id', TEXT_COLUMN, 'model'}),
    ('test', test, {'id', TEXT_COLUMN}),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{name} is missing {sorted(missing)}; inspect the attached files.')
    assert len(frame) > 0, f'{name} is empty'
    assert not frame[list(required)].isna().any().any(), f'Null field in {name}'
    assert frame['id'].str.strip().ne('').all(), f'Empty ID in {name}'
    assert frame['id'].is_unique, f'Duplicate IDs in {name}'
    assert frame[TEXT_COLUMN].str.strip().ne('').all(), f'Empty passage in {name}'

assert train['prompt_id'].str.strip().ne('').all(), 'Missing training prompt IDs'
assert set(train['model']) == set(VALID_LABELS), 'Unexpected or missing training labels; recheck the official label list.'
assert importlib.util.find_spec('pyarrow') is not None, 'Parquet needs pyarrow; use Kaggle or install it locally before running.'

X = train[TEXT_COLUMN].reset_index(drop=True)
y = train['model'].reset_index(drop=True)
groups = train['prompt_id'].reset_index(drop=True)
display(train[['id', 'prompt_id', 'model']].head())
display(y.value_counts().reindex(VALID_LABELS).rename('training_rows').to_frame())
print('Unique prompts:', groups.nunique())
print('Passage lengths in characters:')
display(X.str.len().describe().to_frame('characters'))
print('Repeated training passages:', int(X.duplicated().sum()))

## 3. Model

Words and short phrases capture wording preferences. Character sequences capture
spelling, punctuation, typography and fragments across word boundaries. We keep
stop words because function words can reveal style. Both branches remain sparse.
Feature caps bound memory; the character branch receives slightly more weight.
These are starting choices, not tuned or measured performance claims.

`class_weight='balanced'` increases the penalty for mistakes on rarer training
classes, a reasonable starting point for mean per-class recall. SVM margins are
decision scores, not probabilities; we need only the highest-scoring model label.

In [ ]:
def make_model():
    # A fresh pipeline per fold prevents vocabulary/IDF leakage from validation.
    features = FeatureUnion([
        ('word', TfidfVectorizer(
            analyzer='word', ngram_range=(1, 2), lowercase=False,
            strip_accents=None, min_df=2, max_features=60_000,
            sublinear_tf=True, dtype=np.float32,
        )),
        ('char', TfidfVectorizer(
            analyzer='char', ngram_range=(2, 5), lowercase=False,
            strip_accents=None, min_df=2, max_features=120_000,
            sublinear_tf=True, dtype=np.float32,
        )),
    ], transformer_weights={'word': 0.7, 'char': 1.0}, n_jobs=1)
    return Pipeline([
        ('features', features),
        ('classifier', LinearSVC(
            C=C, class_weight='balanced', dual=True,
            max_iter=10_000, tol=1e-4, random_state=SEED,
        )),
    ])

## 4. Validate on held-out prompts

Responses to a shared prompt must stay together: otherwise training can contain
nearby answers to the validation question. GroupKFold tests transfer to other
prompts. The TF-IDF vocabulary is learned on each training fold only.
We also reject exact repeated passages across a fold boundary; if this triggers,
merge the affected prompt groups before proceeding. Do not ignore the assertion.

Out-of-fold (OOF) predictions give each training row one prediction from a model
that did not train on its prompt. The pooled balanced accuracy is the main local
score. It is an estimate, not a guarantee of leaderboard performance.

In [ ]:
cv_summary = None
if RUN_CV:
    assert 2 <= N_SPLITS <= groups.nunique()
    splitter = GroupKFold(n_splits=N_SPLITS)
    oof = np.full(len(train), '', dtype=object)
    fold_scores = []
    for fold, (fit_idx, val_idx) in enumerate(splitter.split(X, y, groups), start=1):
        assert set(groups.iloc[fit_idx]).isdisjoint(set(groups.iloc[val_idx]))
        assert set(X.iloc[fit_idx]).isdisjoint(set(X.iloc[val_idx])), 'Duplicate text crosses prompts; merge those prompt groups.'
        assert set(y.iloc[fit_idx]) == set(VALID_LABELS), 'A training fold lacks a class.'
        assert set(y.iloc[val_idx]) == set(VALID_LABELS), 'A validation fold lacks a class; reduce N_SPLITS or revise groups.'
        start = time.perf_counter()
        fold_model = make_model()
        fold_model.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        predictions = fold_model.predict(X.iloc[val_idx])
        oof[val_idx] = predictions
        score = balanced_accuracy_score(y.iloc[val_idx], predictions)
        fold_scores.append(float(score))
        print(f'Fold {fold}: balanced accuracy={score:.4f}; seconds={time.perf_counter() - start:.1f}')
        del fold_model
        gc.collect()

    assert np.all(oof != '')
    pooled_score = balanced_accuracy_score(y, oof)
    print(f'Pooled OOF balanced accuracy: {pooled_score:.4f}')
    print(f'Fold mean: {np.mean(fold_scores):.4f}; fold SD: {np.std(fold_scores):.4f}')
    print(f'Uniform-random expected balanced accuracy: {1 / len(VALID_LABELS):.4f}')
    report = pd.DataFrame(classification_report(
        y, oof, labels=VALID_LABELS, output_dict=True, zero_division=0,
    )).T
    display(report.loc[VALID_LABELS, ['precision', 'recall', 'f1-score', 'support']])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'id': train['id'], 'true_model': y, 'predicted_model': oof}).to_csv(
        OUTPUT_DIR / 'oof_predictions.csv', index=False,
    )
    cv_summary = {'pooled_balanced_accuracy': float(pooled_score), 'fold_scores': fold_scores}

    # Each row sums to one; the diagonal is recall for each true model.
    cm = confusion_matrix(y, oof, labels=VALID_LABELS, normalize='true')
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, vmin=0, vmax=1, cmap='Blues')
    ax.set_xticks(range(18), labels=VALID_LABELS, rotation=90)
    ax.set_yticks(range(18), labels=VALID_LABELS)
    ax.set(xlabel='Predicted model', ylabel='True model', title='Prompt-held-out confusion matrix')
    fig.colorbar(im, ax=ax, label='Fraction of true class')
    plt.tight_layout()
    plt.show()
else:
    print('Cross-validation skipped; no local performance estimate.')

## 5. Refit and predict every test row

Train a fresh pipeline on all labeled data after validation. Predict independently
for each passage; do not force equal class counts or one model per inferred prompt.
The hidden test class distribution is unequal. Batching works for the interactive
example and for thousands of hidden rows.

In [ ]:
start = time.perf_counter()
final_model = make_model()
final_model.fit(X, y)
test_predictions = np.concatenate([
    final_model.predict(test[TEXT_COLUMN].iloc[start_row:start_row + BATCH_SIZE])
    for start_row in range(0, len(test), BATCH_SIZE)
])
print(f'Refit and inference: {time.perf_counter() - start:.1f} seconds')
display(pd.Series(test_predictions).value_counts().rename('predicted_rows').to_frame())

## 6. Write and read back the submission

Parquet is a binary table format: a CSV renamed to `.parquet` will not work.
The interactive sample submission can confirm column names, but its example IDs
and row count must never control inference on the hidden rerun.

In [ ]:
def validate_submission(frame, expected_ids):
    assert frame.columns.tolist() == ['id', 'model']
    assert len(frame) == len(expected_ids)
    assert not frame.isna().any().any()
    assert frame['id'].is_unique
    assert frame['id'].str.strip().ne('').all()
    assert frame['id'].tolist() == expected_ids.tolist(), 'IDs or their order changed'
    assert frame['model'].isin(VALID_LABELS).all(), 'Invalid model name'

sample_path = DATA_DIR / 'sample_submission.parquet'
if sample_path.is_file():
    sample = pd.read_parquet(sample_path)
    assert set(sample.columns) == {'id', 'model'}, 'Sample schema differs; recheck requirements.'

submission = pd.DataFrame({'id': test['id'].to_numpy(), 'model': test_predictions})
validate_submission(submission, test['id'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / 'submission.parquet'
submission.to_parquet(submission_path, engine='pyarrow', index=False)
saved = pd.read_parquet(submission_path, engine='pyarrow')
validate_submission(saved, test['id'])
pd.testing.assert_frame_equal(saved, submission)

# A small experiment record helps compare later changes without guessing settings.
metadata = {
    'seed': SEED, 'C': C, 'n_splits': N_SPLITS, 'text_column': TEXT_COLUMN,
    'training_rows': len(train), 'test_rows': len(test), 'cv': cv_summary,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
}
(OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Validated {len(saved):,} predictions: {submission_path}')
display(saved.head())

## 7. Submit on Kaggle

1. Join the competition and review/accept its full rules yourself.
2. Create a competition notebook, import this `.ipynb`, and attach the competition data.
3. Select CPU, turn Internet **off**, and run all cells. Check the validation results
   and the final message confirming `submission.parquet` was read back successfully.
4. Choose **Save Version → Save & Run All**. Wait for that version to finish.
5. Open its **Output** and submit `submission.parquet` to the competition. Kaggle
   will rerun the notebook with hidden test rows before returning a score.

See [Kaggle's submission documentation](https://www.kaggle.com/docs/competitions).
Creating an interactive output file alone does not submit an entry.

For a second experiment, compare character n-gram ranges or C values on the same
prompt folds. Inspect the per-model recall to find weak sibling-model distinctions.
Keep experiments small and record settings. This notebook needs no external models;
any later external data/model additions require their own licensing and rules review.